In [0]:
spark.sql("use catalog e_comm");
spark.sql("use database silver");

In [0]:
spark.sql("create table if not exists e_comm.silver.customers(customer_id string, customer_unique_id string, customer_zip_code_prefix string, customer_city string, customer_state string, start_date timestamp, end_date timestamp, is_active boolean) using delta ")

In [0]:
from pyspark.sql.functions import *
import pandas as pd
import requests
from io import StringIO
from delta.tables import DeltaTable

df_source = spark.read.table("bronze.customers")
df_source = df_source.withColumn("customer_unique_id", substring(col('customer_unique_id'),(length(col('customer_unique_id'))-4),5))
df_target = DeltaTable.forName(spark, "silver.customers")

df_target.alias("t").merge(df_source.alias("s"), "t.customer_id = s.customer_id and t.is_active = True")\
    .whenMatchedUpdate(
        condition = "t.customer_unique_id <> s.customer_unique_id or t.customer_zip_code_prefix <> s.customer_zip_code_prefix or t.customer_city <> s.customer_city or t.customer_state <> s.customer_state",
        set = {
                "t.end_date" : current_timestamp(),
                "t.is_active" : lit(False)
        }).execute()



df_target.alias("t").merge(df_source.alias("s"),"t.customer_id = s.customer_id and t.is_active = True")\
    .whenNotMatchedInsert(
        values = {
            "t.customer_id" : "s.customer_id",
            "t.customer_unique_id" : "s.customer_unique_id",
            "t.customer_zip_code_prefix" : "s.customer_zip_code_prefix",
            "t.customer_city" : "s.customer_city",
            "t.customer_state" : "s.customer_state",
            "t.start_date" : current_timestamp(),
            "t.end_date" : "TIMESTAMP('3000-06-02')",
            "t.is_active" : lit(True)
        }
    ).execute()

In [0]:
%sql
select count(*) from e_comm.silver.customers